# Complex graph with annotations

Seven sequence nodes form two branching regions, with twelve overlapping annotations on both strands. Four annotations span three nodes, following the alternative routes through the graph.

Run all cells with the project's Python environment (`gen` and its Jupyter dependencies installed). This notebook generates its own data and uses Gen's existing `GraphWidget`.

The Python widget currently uses its earlier annotation renderer; the packed annotation bars and spline connectors are not yet wired into that renderer.

In [ ]:
from pathlib import Path
from random import Random
from tempfile import mkdtemp

import gen
from IPython.display import display

## Create the graph

Both branches rejoin before the next split. All links run left to right. The DNA is reproducible synthetic data.

```text
         upper             upper_tail
        /     \           /          \
   first       merge                  last
        \     /           \          /
         lower             lower_tail
```

Keep the temporary repository on disk while using the widget: it reopens the database during navigation.

In [ ]:
node_lengths = {
    "first": 20,
    "upper": 30,
    "lower": 18,
    "merge": 26,
    "upper_tail": 22,
    "lower_tail": 14,
    "last": 20,
}
links = (
    ("first", "upper"),
    ("first", "lower"),
    ("upper", "merge"),
    ("lower", "merge"),
    ("merge", "upper_tail"),
    ("merge", "lower_tail"),
    ("upper_tail", "last"),
    ("lower_tail", "last"),
)

random = Random(17)
sequences = {
    name: "".join(random.choice("ACGT") for _ in range(length))
    for name, length in node_lengths.items()
}
directory = Path(mkdtemp(prefix="gen-complex-annotations-"))
records = ["H\tVN:Z:1.0"]
records.extend(f"S\t{name}\t{sequence}" for name, sequence in sequences.items())
records.extend(f"L\t{source}\t+\t{target}\t+\t0M" for source, target in links)
graph_file = directory / "complex_annotations.gfa"
graph_file.write_text("\n".join(records) + "\n", encoding="utf-8")

repository = gen.Repository(str(directory / ".gen"))
graph = repository.import_gfa(str(graph_file), sample="annotation_demo")
print(f"Created {len(node_lengths)} nodes. Demo data: {directory}")

## Add annotations

Coordinates are zero-based, end-exclusive offsets within each named node. Searching for each feature's sequence gives a native `Locus`, including its node IDs and strand. These annotations are widget overlays, not stored database annotations.

Edit the feature definitions below to try different overlaps, label lengths, or routes through the graph.

In [ ]:
features = (
    ("promoter", "+", (("first", 0, 8),)),
    ("coding", "+", (("first", 10, 20), ("upper", 0, 30), ("merge", 0, 12))),
    ("antisense", "-", (("first", 14, 20), ("lower", 0, 18), ("merge", 0, 8))),
    ("nested", "-", (("upper", 4, 24),)),
    ("binding_site", "+", (("upper", 10, 18),)),
    ("short", "+", (("lower", 3, 11),)),
    ("a_label_too_long_for_either_side", "+", (("lower", 5, 13),)),
    ("bridge", "+", (("merge", 8, 22),)),
    ("isoform_a", "+", (("merge", 18, 26), ("upper_tail", 0, 22), ("last", 0, 8))),
    ("isoform_b", "-", (("merge", 20, 26), ("lower_tail", 0, 14), ("last", 0, 12))),
    ("terminator", "-", (("last", 8, 20),)),
    ("tag", "+", (("upper_tail", 8, 16),)),
)

annotations = []
for name, strand, segments in features:
    sequence = "".join(sequences[node][start:end] for node, start, end in segments)
    if strand == "-":
        sequence = sequence.translate(str.maketrans("ACGT", "TGCA"))[::-1]
    matches = [locus for locus in graph.search(sequence, "dna") if locus.strand == strand]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one locus for {name!r}, found {len(matches)}")
    annotation = gen.Annotation(matches[0], name)
    if len(annotation.segments) != len(segments):
        raise RuntimeError(f"Unexpected fragment count for {name!r}")
    annotations.append(annotation)
    print(f"{name}: {len(annotation)} bp, {len(annotation.segments)} fragment(s), strand {strand}")

## Explore the interactive graph

The existing widget provides zoom, pan, and annotation navigation. This starts at full sequence detail with wider spacing between nodes.

In [ ]:
widget = graph.plot(rows=36, cols=160, detail="full")
widget.add_annotation_track(annotations, name="Synthetic features")
widget.zoom_in()
widget.zoom_in()
display(widget)

You can also navigate programmatically in another cell:

```python
coding = next(annotation for annotation in annotations if annotation.name == "coding")
widget.go_to(coding)
display(widget)
```

Use `widget.zoom_out()`, `widget.zoom_in()`, or `widget.scroll_right()` to explore. Call `display(widget)` afterward to show the changed view; each displayed widget has its own independent controls.